# PCOS Clinical EDA and Phenotype Profiling

## Introduction
This notebook is performing the flagship exploratory data analysis for the cleaned clinical PCOS dataset. The analysis is focusing on the clinical phenotype of PCOS, with special attention to features that may later support accessible and low-burden screening.

The notebook is using `cleaned_data/PCOS_full_cleaned.csv` as the primary analytical source. The clinical table is containing both routine variables and selected invasive markers, so the notebook is prioritizing non-invasive signals first and then using ovarian and morphological variables only where they are helping the biological interpretation.

## Research Context
The broader MSc study is investigating whether PCOS can be profiled and later predicted using routine clinical, symptom-based, and low-burden indicators. This notebook is therefore asking structured research questions about body composition, menstrual pattern variables, visible symptom clusters, cardiometabolic proxies, and ovarian morphology.

Every analytical question in this notebook is following the same pattern:
- a markdown question is introducing the analysis
- a summary table is being printed before the visual
- a figure is then being generated and saved to `images/eda/clinical/`
- an insight cell is then documenting why the pattern matters clinically

The notebook is not performing model training, heart-dataset comparison, optimization, or explainability work. It is only building the clinical evidence base that will support those later phases.


## Reproducibility Setup and Path Configuration

This section is importing the required libraries, resolving the project root safely, and preparing the figure directory for this clinical EDA workflow.


In [ ]:
# Importing the libraries is supporting reproducible data handling, plotting, and notebook display.
from pathlib import Path
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Fixing the random seed is keeping any sampling or jitter behavior stable across reruns.
np.random.seed(42)

# Configuring the plotting theme is keeping the visuals readable and consistent across the notebook.
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# Resolving the project root is keeping the notebook runnable from the repository root or the notebook folder.
def resolve_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for candidate in candidates:
        if (candidate / "cleaned_data").exists() and (candidate / "images").exists():
            return candidate
    return current

PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "cleaned_data" / "PCOS_full_cleaned.csv"
IMAGE_DIR = PROJECT_ROOT / "images" / "eda" / "clinical"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Defining the palette is locking the target-color identity for the full PCOS EDA series.
PCOS_LABEL_ORDER = ["PCOS Negative", "PCOS Positive"]
PCOS_LABEL_PALETTE = {
    "PCOS Negative": "#3b82f6",
    "PCOS Positive": "#e76f51",
}
ACCENT_PALETTE = ["#2a9d8f", "#e9c46a", "#6d597a", "#264653"]

# Saving figures through a helper is keeping the export path consistent and descriptive.
def save_figure(fig: plt.Figure, slug: str) -> Path:
    output_path = IMAGE_DIR / slug
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return output_path

# Building grouped numeric summaries is standardizing the table-first workflow used throughout the notebook.
def grouped_numeric_summary(data: pd.DataFrame, feature: str) -> pd.DataFrame:
    summary = (
        data.groupby("pcos_label")[feature]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={"count": "n"})
        .round(3)
        .reset_index()
    )
    return summary

# Building grouped prevalence summaries is supporting the binary symptom analyses.
def grouped_binary_prevalence(data: pd.DataFrame, feature: str) -> pd.DataFrame:
    summary = (
        data.groupby("pcos_label")[feature]
        .agg(["count", "sum", "mean"])
        .rename(columns={"count": "n", "sum": "positive_count", "mean": "prevalence"})
        .reset_index()
    )
    summary["prevalence_pct"] = (summary["prevalence"] * 100).round(1)
    return summary[["pcos_label", "n", "positive_count", "prevalence_pct"]]

# Plotting numeric distributions through one helper is keeping the comparison style consistent across questions.
def plot_numeric_by_target(
    data: pd.DataFrame,
    feature: str,
    ylabel: str,
    title: str,
    slug: str,
    kind: str = "box",
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    if kind == "violin":
        sns.violinplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            palette=PCOS_LABEL_PALETTE,
            cut=0,
            inner=None,
            ax=ax,
        )
        sns.stripplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            color="#264653",
            alpha=0.30,
            size=3,
            jitter=0.20,
            ax=ax,
        )
    elif kind == "kde":
        for label in PCOS_LABEL_ORDER:
            subset = data.loc[data["pcos_label"] == label, feature].dropna()
            sns.kdeplot(
                subset,
                fill=True,
                alpha=0.24,
                linewidth=2,
                label=label,
                color=PCOS_LABEL_PALETTE[label],
                ax=ax,
            )
        ax.legend(frameon=True)
    else:
        sns.boxplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            palette=PCOS_LABEL_PALETTE,
            ax=ax,
        )
        sns.stripplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            color="#264653",
            alpha=0.30,
            size=3,
            jitter=0.20,
            ax=ax,
        )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    save_figure(fig, slug)
    plt.show()

# Plotting prevalence bars through one helper is simplifying the repeated binary-feature visuals.
def plot_binary_prevalence(summary: pd.DataFrame, title: str, slug: str, ylabel: str = "Prevalence (%)") -> None:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.barplot(
        data=summary,
        x="pcos_label",
        y="prevalence_pct",
        order=PCOS_LABEL_ORDER,
        palette=PCOS_LABEL_PALETTE,
        ax=ax,
    )
    for patch in ax.patches:
        height = patch.get_height()
        ax.annotate(
            f"{height:.1f}%",
            (patch.get_x() + patch.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=11,
            xytext=(0, 6),
            textcoords="offset points",
        )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    save_figure(fig, slug)
    plt.show()


## Data Loading and Feature Preparation

This section is loading the cleaned clinical table, verifying that the expected columns are present, and creating a small set of derived features that will support later EDA questions.


In [ ]:
# Loading the cleaned clinical dataset is bringing the analysis-ready table into memory for EDA.
df = pd.read_csv(DATA_PATH)

# Verifying the expected schema is protecting the notebook from silent upstream changes.
required_columns = [
    "pcos_y_n",
    "age_yrs",
    "weight_kg",
    "height_cm",
    "bmi",
    "pulse_rate_bpm",
    "respiratory_rate_breaths_min",
    "hb_g_dl",
    "cycle_regularity_code",
    "cycle_length_days",
    "waist_hip_ratio",
    "rbs_mg_dl",
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "follicle_no_left",
    "follicle_no_right",
    "avg_follicle_size_left_mm",
    "avg_follicle_size_right_mm",
    "endometrium_mm",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Standardizing the target and binary fields is keeping the plotting logic explicit and stable.
binary_columns = [
    "pcos_y_n",
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
]
for column in binary_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype(int)

numeric_columns = [column for column in required_columns if column not in binary_columns]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Creating readable label columns is improving the presentation of repeated plots.
df["pcos_label"] = df["pcos_y_n"].map({0: "PCOS Negative", 1: "PCOS Positive"})
for column in [
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
]:
    df[f"{column}_label"] = df[column].map({0: "No", 1: "Yes"})

# Creating derived features is supporting the later research questions without modifying the saved cleaned table.
df["total_follicle_count"] = df["follicle_no_left"] + df["follicle_no_right"]
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25.0, 30.0, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    include_lowest=True,
)
df["bmi_overweight_flag"] = (df["bmi"] >= 25).astype(int)
df["low_exercise_flag"] = (1 - df["regular_exercise_y_n"]).astype(int)
df["non_invasive_burden_score"] = (
    df["weight_gain_y_n"]
    + df["hair_growth_y_n"]
    + df["skin_darkening_y_n"]
    + df["hair_loss_y_n"]
    + df["pimples_y_n"]
    + df["fast_food_y_n"]
    + df["bmi_overweight_flag"]
    + df["low_exercise_flag"]
)

# Displaying a compact preview is confirming that the derived fields are present before the question bank starts.
display(df.head())


## Dataset Overview and Target Distribution

This section is establishing the analytical baseline for the notebook before the feature-by-feature profiling begins.


## Question 1

### What is the cleaned clinical dataset size, schema, and target balance?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the overview tables is summarizing the structure and target balance before plotting.
overview_q01 = pd.DataFrame(
    {
        "metric": ["row_count", "column_count", "numeric_column_count", "missing_cells"],
        "value": [
            df.shape[0],
            df.shape[1],
            int(df.select_dtypes(include=np.number).shape[1]),
            int(df.isna().sum().sum()),
        ],
    }
)
schema_q01 = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values})
target_q01 = (
    df["pcos_label"]
    .value_counts()
    .reindex(PCOS_LABEL_ORDER)
    .rename_axis("pcos_label")
    .reset_index(name="count")
)
target_q01["percentage"] = (100 * target_q01["count"] / target_q01["count"].sum()).round(1)

display(overview_q01)
display(schema_q01)
display(target_q01)


In [ ]:
# Plotting the target distribution is showing the class balance that later modeling work will inherit.
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=target_q01,
    x="pcos_label",
    y="count",
    order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(
        f"{int(height)}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        xytext=(0, 6),
        textcoords="offset points",
    )
ax.set_title("Question 1: Cleaned Clinical PCOS Target Distribution")
ax.set_xlabel("")
ax.set_ylabel("Participant Count")
save_figure(fig, "q01_clinical_target_distribution.png")
plt.show()


### Insight


    The cleaned clinical dataset is containing 541 participants, with 364 PCOS-negative cases and 177 PCOS-positive cases. This class balance is not being extreme, but it is still making false negatives clinically important because each missed positive case represents a participant who may remain without timely screening attention. The target distribution is therefore providing context for every later interpretation in this notebook.


## Question 2

### What is the mean and median phenotype profile of PCOS-positive versus PCOS-negative participants across core clinical features?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building a wide phenotype summary is comparing the central tendency of the core clinical variables by PCOS status.
core_profile_features = [
    "age_yrs",
    "weight_kg",
    "bmi",
    "waist_hip_ratio",
    "pulse_rate_bpm",
    "respiratory_rate_breaths_min",
    "hb_g_dl",
    "cycle_length_days",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "rbs_mg_dl",
    "total_follicle_count",
]
profile_mean_q02 = df.groupby("pcos_label")[core_profile_features].mean().round(3).T
profile_median_q02 = df.groupby("pcos_label")[core_profile_features].median().round(3).T
profile_q02 = pd.concat({"mean": profile_mean_q02, "median": profile_median_q02}, axis=1)
display(profile_q02)


In [ ]:
# Plotting the phenotype heatmap is showing where the group-level clinical profile is shifting most clearly.
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    profile_mean_q02,
    annot=True,
    fmt=".2f",
    cmap="RdYlBu_r",
    linewidths=0.5,
    cbar_kws={"label": "Group Mean"},
    ax=ax,
)
ax.set_title("Question 2: Group Mean Clinical Phenotype Profile")
ax.set_xlabel("")
ax.set_ylabel("Feature")
save_figure(fig, "q02_core_phenotype_heatmap.png")
plt.show()


### Insight


    The phenotype profile is already showing that the clearest upward shifts in the PCOS-positive group are clustering around weight, BMI, symptom-linked burden, and follicle counts, while some general vital signs are moving much less. This pattern is suggesting that body composition, menstrual disruption, and ovarian morphology are carrying more discriminatory signal than routine hemodynamic measures alone.


## Non-Invasive Continuous Feature Analysis

This section is profiling the continuous features that can contribute to low-burden or routine PCOS screening.


## Question 3

### Does age differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `age_yrs` before plotting.
summary_q03 = grouped_numeric_summary(df, "age_yrs")
display(summary_q03)


In [ ]:
# Plotting the grouped distribution is showing how `age_yrs` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="age_yrs",
    ylabel="Age (years)",
    title="Question 3: Does age differ by PCOS status?",
    slug="q03_age_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

This figure is showing whether the age distribution is shifting materially between the two PCOS groups. The small separation already visible in the cleaned cohort is suggesting that age alone is not behaving like a strong standalone discriminator, which matters because it should probably act as a context feature rather than a dominant screening signal.


## Question 4

### Does weight differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `weight_kg` before plotting.
summary_q04 = grouped_numeric_summary(df, "weight_kg")
display(summary_q04)


In [ ]:
# Plotting the grouped distribution is showing how `weight_kg` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="weight_kg",
    ylabel="Weight (kg)",
    title="Question 4: Does weight differ by PCOS status?",
    slug="q04_weight_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

The weight distribution is showing a visible upward shift in the PCOS-positive cohort. That pattern is supporting the idea that higher body mass is clustering with PCOS in this sample, which matters because weight is easy to obtain and may strengthen non-invasive screening when combined with more specific symptoms.


## Question 5

### Does BMI differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `bmi` before plotting.
summary_q05 = grouped_numeric_summary(df, "bmi")
display(summary_q05)


In [ ]:
# Plotting the grouped distribution is showing how `bmi` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="bmi",
    ylabel="Body Mass Index",
    title="Question 5: Does BMI differ by PCOS status?",
    slug="q05_bmi_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

BMI is showing one of the clearer non-invasive shifts in the cleaned clinical cohort, with the PCOS-positive group carrying a higher central tendency. This matters clinically because adiposity is often traveling with insulin resistance and ovulatory dysfunction, making BMI a strong candidate for later routine-feature modeling.


## Question 6

### Does waist-hip ratio differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `waist_hip_ratio` before plotting.
summary_q06 = grouped_numeric_summary(df, "waist_hip_ratio")
display(summary_q06)


In [ ]:
# Plotting the grouped distribution is showing how `waist_hip_ratio` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="waist_hip_ratio",
    ylabel="Waist-Hip Ratio",
    title="Question 6: Does waist-hip ratio differ by PCOS status?",
    slug="q06_waist_hip_ratio_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Waist-hip ratio is showing only a modest separation relative to BMI, which is suggesting that this anthropometric marker may add nuance but not dominate the phenotype story by itself. It is still worth retaining because central fat distribution can signal metabolic stress even when total BMI is only moderately elevated.


## Question 7

### Does pulse rate differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `pulse_rate_bpm` before plotting.
summary_q07 = grouped_numeric_summary(df, "pulse_rate_bpm")
display(summary_q07)


In [ ]:
# Plotting the grouped distribution is showing how `pulse_rate_bpm` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="pulse_rate_bpm",
    ylabel="Pulse Rate (bpm)",
    title="Question 7: Does pulse rate differ by PCOS status?",
    slug="q07_pulse_rate_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Pulse rate is showing little visual separation between the two groups, which is suggesting a weaker direct relationship with PCOS in this cohort. This matters because weak vital-sign movement can help us avoid overvaluing features that are easy to collect but clinically noisy.


## Question 8

### Does respiratory rate differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `respiratory_rate_breaths_min` before plotting.
summary_q08 = grouped_numeric_summary(df, "respiratory_rate_breaths_min")
display(summary_q08)


In [ ]:
# Plotting the grouped distribution is showing how `respiratory_rate_breaths_min` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="respiratory_rate_breaths_min",
    ylabel="Respiratory Rate (breaths/min)",
    title="Question 8: Does respiratory rate differ by PCOS status?",
    slug="q08_respiratory_rate_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Respiratory rate is appearing fairly stable across groups, which is suggesting that it is not a major PCOS phenotype marker in this dataset. That weak movement is useful in itself because it helps distinguish broad physiological context variables from more syndrome-specific features.


## Question 9

### Does hemoglobin differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `hb_g_dl` before plotting.
summary_q09 = grouped_numeric_summary(df, "hb_g_dl")
display(summary_q09)


In [ ]:
# Plotting the grouped distribution is showing how `hb_g_dl` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="hb_g_dl",
    ylabel="Hemoglobin (g/dL)",
    title="Question 9: Does hemoglobin differ by PCOS status?",
    slug="q09_hb_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Hemoglobin is showing only a light shift between groups, so it is likely acting as a background physiological marker rather than a headline discriminator. This pattern is encouraging a cautious interpretation of blood indices unless they meaningfully improve multivariable performance later.


## Question 10

### Does the dataset's recorded cycle-length measure differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `cycle_length_days` before plotting.
summary_q10 = grouped_numeric_summary(df, "cycle_length_days")
display(summary_q10)


In [ ]:
# Plotting the grouped distribution is showing how `cycle_length_days` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="cycle_length_days",
    ylabel="Recorded Cycle-Length Measure",
    title="Question 10: Does the dataset's recorded cycle-length measure differ by PCOS status?",
    slug="q10_cycle_length_measure_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

The recorded cycle-length measure is shifting across PCOS status, but it is behaving like a short ordinal clinical record rather than a literal 21-to-35-day cycle field. This still matters because menstrual-pattern recording is clinically relevant, but the variable should be interpreted as a recorded measure rather than as a textbook day count.


## Question 11

### Does cycle_regularity_code differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `cycle_regularity_code` before plotting.
summary_q11 = grouped_numeric_summary(df, "cycle_regularity_code")
display(summary_q11)


In [ ]:
# Plotting the grouped distribution is showing how `cycle_regularity_code` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="cycle_regularity_code",
    ylabel="Cycle Regularity Code",
    title="Question 11: Does cycle_regularity_code differ by PCOS status?",
    slug="q11_cycle_regularity_code_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

The cycle-regularity code is showing a stronger group shift than many of the routine vital signs, which is reinforcing menstrual irregularity as a core PCOS signal. Because the coding scheme is ordinal and not yet mapped to human labels, it should be handled as a recorded code until a documented mapping is introduced.


## Question 12

### Does systolic blood pressure differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `systolic_bp_mmhg` before plotting.
summary_q12 = grouped_numeric_summary(df, "systolic_bp_mmhg")
display(summary_q12)


In [ ]:
# Plotting the grouped distribution is showing how `systolic_bp_mmhg` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="systolic_bp_mmhg",
    ylabel="Systolic Blood Pressure (mmHg)",
    title="Question 12: Does systolic blood pressure differ by PCOS status?",
    slug="q12_systolic_bp_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Systolic blood pressure is showing only a limited shift between groups in the clinical cohort. This is suggesting that blood pressure may contribute more to cardiometabolic context than to direct PCOS discrimination when used alone.


## Question 13

### Does diastolic blood pressure differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `diastolic_bp_mmhg` before plotting.
summary_q13 = grouped_numeric_summary(df, "diastolic_bp_mmhg")
display(summary_q13)


In [ ]:
# Plotting the grouped distribution is showing how `diastolic_bp_mmhg` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="diastolic_bp_mmhg",
    ylabel="Diastolic Blood Pressure (mmHg)",
    title="Question 13: Does diastolic blood pressure differ by PCOS status?",
    slug="q13_diastolic_bp_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Diastolic blood pressure is remaining tightly clustered across the two groups, which is again pointing to a weak standalone relationship with PCOS status in this dataset. This observation will still matter later when the cardiometabolic comparison notebook is examining broader risk context.


## Question 14

### Does random blood sugar differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying the distribution of `rbs_mg_dl` before plotting.
summary_q14 = grouped_numeric_summary(df, "rbs_mg_dl")
display(summary_q14)


In [ ]:
# Plotting the grouped distribution is showing how `rbs_mg_dl` is shifting across PCOS status.
plot_numeric_by_target(
    data=df,
    feature="rbs_mg_dl",
    ylabel="Random Blood Sugar (mg/dL)",
    title="Question 14: Does random blood sugar differ by PCOS status?",
    slug="q14_rbs_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Random blood sugar is showing a slight upward shift and a wider upper tail in the PCOS-positive group. That pattern is clinically relevant because glycemic stress is often co-traveling with PCOS, even when blood sugar alone is not cleanly separating the groups.


## Binary Symptom and Behavior Feature Analysis

This section is profiling the self-reported and clinically observed binary features that may be especially valuable for low-cost screening.


## Question 15

### Does weight gain prevalence differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `weight_gain_y_n` is present in each PCOS group.
summary_q15 = grouped_binary_prevalence(df, "weight_gain_y_n")
display(summary_q15)


In [ ]:
# Plotting the grouped prevalence is showing how `weight_gain_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q15,
    title="Question 15: Weight Gain Prevalence by PCOS Status",
    slug="q15_weight_gain_prevalence.png",
)


### Insight

The weight-gain signal is showing a large prevalence gap between the two groups, with the PCOS-positive cohort carrying a much higher burden. This matters because a strong symptom-prevalence shift in an easy-to-report feature is valuable for accessible screening workflows.


## Question 16

### Does hair growth prevalence differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `hair_growth_y_n` is present in each PCOS group.
summary_q16 = grouped_binary_prevalence(df, "hair_growth_y_n")
display(summary_q16)


In [ ]:
# Plotting the grouped prevalence is showing how `hair_growth_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q16,
    title="Question 16: Hair Growth Prevalence by PCOS Status",
    slug="q16_hair_growth_prevalence.png",
)


### Insight

Hair growth is showing a pronounced prevalence difference, which is supporting its relevance as an androgen-linked symptom marker. This is important because visible hyperandrogenic features can contribute strong non-invasive signal when laboratory testing is not immediately available.


## Question 17

### Does skin darkening prevalence differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `skin_darkening_y_n` is present in each PCOS group.
summary_q17 = grouped_binary_prevalence(df, "skin_darkening_y_n")
display(summary_q17)


In [ ]:
# Plotting the grouped prevalence is showing how `skin_darkening_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q17,
    title="Question 17: Skin Darkening Prevalence by PCOS Status",
    slug="q17_skin_darkening_prevalence.png",
)


### Insight

Skin darkening is showing one of the sharpest prevalence differences in the clinical dataset. That pattern is clinically meaningful because acanthosis-like darkening can reflect insulin resistance and metabolic burden, both of which frequently cluster with PCOS.


## Question 18

### Does hair loss prevalence differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `hair_loss_y_n` is present in each PCOS group.
summary_q18 = grouped_binary_prevalence(df, "hair_loss_y_n")
display(summary_q18)


In [ ]:
# Plotting the grouped prevalence is showing how `hair_loss_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q18,
    title="Question 18: Hair Loss Prevalence by PCOS Status",
    slug="q18_hair_loss_prevalence.png",
)


### Insight

Hair loss is showing a noticeable but more moderate prevalence gap than some of the stronger symptom variables. This suggests that it may still contribute useful signal, although it is likely to be less specific than hair growth or skin darkening.


## Question 19

### Does pimples prevalence differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `pimples_y_n` is present in each PCOS group.
summary_q19 = grouped_binary_prevalence(df, "pimples_y_n")
display(summary_q19)


In [ ]:
# Plotting the grouped prevalence is showing how `pimples_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q19,
    title="Question 19: Pimples Prevalence by PCOS Status",
    slug="q19_pimples_prevalence.png",
)


### Insight

Pimples are showing a clear upward prevalence shift in the PCOS-positive group. This matters because acne-related features often reflect androgen activity, but the signal may overlap with other visible symptoms and therefore needs redundancy checking later in the notebook.


## Question 20

### Does fast-food behavior differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `fast_food_y_n` is present in each PCOS group.
summary_q20 = grouped_binary_prevalence(df, "fast_food_y_n")
display(summary_q20)


In [ ]:
# Plotting the grouped prevalence is showing how `fast_food_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q20,
    title="Question 20: Fast Food Prevalence by PCOS Status",
    slug="q20_fast_food_prevalence.png",
)


### Insight

Fast-food behavior is showing a strong prevalence difference, which may be reflecting a broader lifestyle or metabolic risk pattern rather than a syndrome-specific biological marker. This makes the variable interesting, but it should be interpreted carefully because behavior variables can be confounded by social and reporting effects.


## Question 21

### Does regular exercise differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `regular_exercise_y_n` is present in each PCOS group.
summary_q21 = grouped_binary_prevalence(df, "regular_exercise_y_n")
display(summary_q21)


In [ ]:
# Plotting the grouped prevalence is showing how `regular_exercise_y_n` is concentrating by PCOS status.
plot_binary_prevalence(
    summary=summary_q21,
    title="Question 21: Regular Exercise Prevalence by PCOS Status",
    slug="q21_regular_exercise_prevalence.png",
)


### Insight

Regular exercise is showing only a limited difference between groups in this cohort. That weaker separation suggests it may be more useful as a contextual lifestyle modifier than as a primary screening feature by itself.


## Reproductive, Cardiometabolic, and Ovarian Structure Questions

This section is extending the clinical EDA into ovarian morphology and other biologically relevant markers that deepen the phenotype story.


## Question 22

### Do follicle counts on the left ovary differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `follicle_no_left` by PCOS status before plotting.
summary_q22 = grouped_numeric_summary(df, "follicle_no_left")
display(summary_q22)


In [ ]:
# Plotting the grouped distribution is showing how `follicle_no_left` is moving across the PCOS classes.
plot_numeric_by_target(
    data=df,
    feature="follicle_no_left",
    ylabel="Left Follicle Count",
    title="Question 22: Do follicle counts on the left ovary differ by PCOS status?",
    slug="q22_follicle_left_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Left-ovary follicle counts are showing a marked upward shift in the PCOS-positive group. This is clinically coherent because increased follicle burden is closely tied to the ovarian morphology associated with PCOS.


## Question 23

### Do follicle counts on the right ovary differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `follicle_no_right` by PCOS status before plotting.
summary_q23 = grouped_numeric_summary(df, "follicle_no_right")
display(summary_q23)


In [ ]:
# Plotting the grouped distribution is showing how `follicle_no_right` is moving across the PCOS classes.
plot_numeric_by_target(
    data=df,
    feature="follicle_no_right",
    ylabel="Right Follicle Count",
    title="Question 23: Do follicle counts on the right ovary differ by PCOS status?",
    slug="q23_follicle_right_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Right-ovary follicle counts are also showing a strong upward shift among PCOS-positive participants. Seeing the pattern on both sides is strengthening the interpretation that ovarian morphology is one of the clearest differentiators in the cleaned clinical cohort.


## Question 24

### Do average follicle sizes differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `avg_follicle_size_right_mm` by PCOS status before plotting.
summary_q24 = grouped_numeric_summary(df, "avg_follicle_size_right_mm")
display(summary_q24)


In [ ]:
# Plotting the grouped distribution is showing how `avg_follicle_size_right_mm` is moving across the PCOS classes.
plot_numeric_by_target(
    data=df,
    feature="avg_follicle_size_right_mm",
    ylabel="Average Right Follicle Size (mm)",
    title="Question 24: Do average follicle sizes differ by PCOS status?",
    slug="q24_avg_follicle_size_vs_pcos_violin.png",
    kind="violin",
)


### Insight

Average follicle size is showing a more modest group separation than follicle count, which is suggesting that count may be the more informative ovarian measure here. This matters because it helps prioritize which invasive features are truly contributing unique signal.


## Question 25

### Does endometrium thickness differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `endometrium_mm` by PCOS status before plotting.
summary_q25 = grouped_numeric_summary(df, "endometrium_mm")
display(summary_q25)


In [ ]:
# Plotting the grouped distribution is showing how `endometrium_mm` is moving across the PCOS classes.
plot_numeric_by_target(
    data=df,
    feature="endometrium_mm",
    ylabel="Endometrium Thickness (mm)",
    title="Question 25: Does endometrium thickness differ by PCOS status?",
    slug="q25_endometrium_vs_pcos_boxplot.png",
    kind="box",
)


### Insight

Endometrium thickness is showing only a moderate shift between groups, which suggests it may offer supplementary biological context rather than headline discrimination. This is useful for interpretation, but it should likely sit below symptom burden and follicle count in later priority lists.


## Question 26

### Are follicle counts higher among participants with skin darkening?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped follicle summary is showing how ovarian burden changes across `skin_darkening_y_n` and PCOS status.
summary_q26 = (
    df.groupby(["pcos_label", "skin_darkening_y_n"])["total_follicle_count"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={"count": "n"})
    .round(3)
    .reset_index()
)
summary_q26["skin_darkening_y_n_label"] = summary_q26["skin_darkening_y_n"].map({0: "No", 1: "Yes"})
display(summary_q26)


In [ ]:
# Plotting the follicle burden by symptom status is showing whether the symptom is aligning with ovarian morphology.
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="skin_darkening_y_n_label",
    y="total_follicle_count",
    hue="pcos_label",
    order=["No", "Yes"],
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 26: Total Follicle Count by Skin Darkening and PCOS Status")
ax.set_xlabel("Skin Darkening Reported")
ax.set_ylabel("Total Follicle Count")
ax.legend(title="")
save_figure(fig, "q26_total_follicles_by_skin_darkening.png")
plt.show()


### Insight

This figure is showing whether visible metabolic-skin changes are clustering with a heavier follicle burden. A higher follicle count among participants with skin darkening would strengthen the link between external symptom expression and deeper ovarian phenotype.


## Question 27

### Are follicle counts higher among participants with hair growth?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped follicle summary is showing how ovarian burden changes across `hair_growth_y_n` and PCOS status.
summary_q27 = (
    df.groupby(["pcos_label", "hair_growth_y_n"])["total_follicle_count"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={"count": "n"})
    .round(3)
    .reset_index()
)
summary_q27["hair_growth_y_n_label"] = summary_q27["hair_growth_y_n"].map({0: "No", 1: "Yes"})
display(summary_q27)


In [ ]:
# Plotting the follicle burden by symptom status is showing whether the symptom is aligning with ovarian morphology.
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="hair_growth_y_n_label",
    y="total_follicle_count",
    hue="pcos_label",
    order=["No", "Yes"],
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 27: Total Follicle Count by Hair Growth and PCOS Status")
ax.set_xlabel("Hair Growth Reported")
ax.set_ylabel("Total Follicle Count")
ax.legend(title="")
save_figure(fig, "q27_total_follicles_by_hair_growth.png")
plt.show()


### Insight

This comparison is showing whether androgen-linked hair growth is traveling with higher follicle burden. If the follicle distribution is shifting upward with hair growth, the symptom is reinforcing its role as a clinically meaningful surface marker of a deeper PCOS phenotype.


## Question 28

### Are follicle counts higher among participants with weight gain?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped follicle summary is showing how ovarian burden changes across `weight_gain_y_n` and PCOS status.
summary_q28 = (
    df.groupby(["pcos_label", "weight_gain_y_n"])["total_follicle_count"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={"count": "n"})
    .round(3)
    .reset_index()
)
summary_q28["weight_gain_y_n_label"] = summary_q28["weight_gain_y_n"].map({0: "No", 1: "Yes"})
display(summary_q28)


In [ ]:
# Plotting the follicle burden by symptom status is showing whether the symptom is aligning with ovarian morphology.
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="weight_gain_y_n_label",
    y="total_follicle_count",
    hue="pcos_label",
    order=["No", "Yes"],
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 28: Total Follicle Count by Weight Gain and PCOS Status")
ax.set_xlabel("Weight Gain Reported")
ax.set_ylabel("Total Follicle Count")
ax.legend(title="")
save_figure(fig, "q28_total_follicles_by_weight_gain.png")
plt.show()


### Insight

This question is linking metabolic burden and ovarian morphology directly. A heavier follicle burden among participants who report weight gain would support the idea that body-composition stress and ovarian change are clustering in the same high-risk subgroup.


## Joint Shift and Redundancy Analysis

This section is examining multivariate movement and feature overlap so that later modeling work can prioritize strong but non-redundant signals.


## Question 29

### Are BMI and waist-hip ratio jointly shifted in the PCOS-positive cohort?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the joint summary is quantifying the anthropometric shift before drawing the two-dimensional scatter.
summary_q29 = (
    df.groupby("pcos_label")[["bmi", "waist_hip_ratio"]]
    .agg(["mean", "median", "std"])
    .round(3)
)
display(summary_q29)


In [ ]:
# Plotting the joint anthropometric space is showing where the two groups are clustering together and apart.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="bmi",
    y="waist_hip_ratio",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.75,
    s=70,
    ax=ax,
)
ax.set_title("Question 29: BMI and Waist-Hip Ratio by PCOS Status")
ax.set_xlabel("Body Mass Index")
ax.set_ylabel("Waist-Hip Ratio")
ax.legend(title="")
save_figure(fig, "q29_bmi_waist_hip_joint_scatter.png")
plt.show()


### Insight


    The anthropometric scatter is showing that BMI is contributing more visible separation than waist-hip ratio in this cohort, even though both are pointing toward adiposity. This matters because it suggests BMI may remain the higher-priority routine feature, while waist-hip ratio may act as a smaller complementary marker rather than a dominant one.


## Question 30

### Are blood pressure and random blood sugar jointly shifted in the PCOS-positive cohort?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the cardiometabolic summary is quantifying the shared movement of blood pressure and glucose-related variables.
summary_q30 = (
    df.groupby("pcos_label")[["systolic_bp_mmhg", "diastolic_bp_mmhg", "rbs_mg_dl"]]
    .agg(["mean", "median", "std"])
    .round(3)
)
display(summary_q30)


In [ ]:
# Plotting the joint cardiometabolic space is showing whether the PCOS-positive cohort is sitting further along a risk-leaning profile.
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="systolic_bp_mmhg",
    y="rbs_mg_dl",
    hue="pcos_label",
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    alpha=0.75,
    s=70,
    ax=ax,
)
ax.set_title("Question 30: Systolic Blood Pressure and Random Blood Sugar by PCOS Status")
ax.set_xlabel("Systolic Blood Pressure (mmHg)")
ax.set_ylabel("Random Blood Sugar (mg/dL)")
ax.legend(title="")
save_figure(fig, "q30_bp_rbs_joint_scatter.png")
plt.show()


### Insight


    The joint cardiometabolic scatter is showing overlap between the two groups, but it is also allowing the wider upper tail of random blood sugar in the PCOS-positive cohort to remain visible. This suggests that blood pressure and glycemic markers may be contributing more as metabolic-context features than as sharp standalone PCOS separators.


## Question 31

### Which continuous variables show the strongest Spearman correlation with pcos_y_n?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the target-correlation table is ranking the monotonic relationship between each continuous feature and PCOS status.
continuous_features_q31 = [
    "age_yrs",
    "weight_kg",
    "bmi",
    "waist_hip_ratio",
    "pulse_rate_bpm",
    "respiratory_rate_breaths_min",
    "hb_g_dl",
    "cycle_length_days",
    "cycle_regularity_code",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "rbs_mg_dl",
    "follicle_no_left",
    "follicle_no_right",
    "avg_follicle_size_left_mm",
    "avg_follicle_size_right_mm",
    "endometrium_mm",
    "total_follicle_count",
]
summary_q31 = (
    df[continuous_features_q31 + ["pcos_y_n"]]
    .corr(method="spearman")["pcos_y_n"]
    .drop("pcos_y_n")
    .sort_values(key=lambda series: series.abs(), ascending=False)
    .rename("spearman_r")
    .reset_index()
    .rename(columns={"index": "feature"})
)
summary_q31["abs_spearman_r"] = summary_q31["spearman_r"].abs().round(3)
summary_q31["spearman_r"] = summary_q31["spearman_r"].round(3)
display(summary_q31)


In [ ]:
# Plotting the target-correlation ranking is highlighting which continuous variables deserve the most modeling attention.
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(
    data=summary_q31,
    y="feature",
    x="spearman_r",
    palette="viridis",
    ax=ax,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_title("Question 31: Spearman Correlation with PCOS Status")
ax.set_xlabel("Spearman Correlation")
ax.set_ylabel("Feature")
save_figure(fig, "q31_continuous_spearman_with_target.png")
plt.show()


### Insight


    This ranking is showing that ovarian and adiposity-linked variables are carrying stronger monotonic alignment with PCOS status than general vital signs. That matters because later modeling should be emphasizing features that are both clinically plausible and empirically aligned with the target, rather than keeping weak routine variables simply because they are available.


## Question 32

### Which symptoms show the strongest Spearman correlation with each other?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the symptom-correlation matrix is quantifying redundancy among the binary symptom and behavior features.
symptom_features_q32 = [
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
]
summary_q32 = df[symptom_features_q32].corr(method="spearman").round(3)
display(summary_q32)


In [ ]:
# Plotting the symptom-correlation heatmap is showing where overlapping signal may reduce the need for redundant features.
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    summary_q32,
    annot=True,
    fmt=".2f",
    cmap="mako",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Question 32: Symptom Spearman Correlation Matrix")
save_figure(fig, "q32_symptom_spearman_heatmap.png")
plt.show()


### Insight


    The symptom correlation matrix is showing where visible or behavior-linked variables are moving together and potentially carrying overlapping signal. This matters because a later model may not benefit from keeping every correlated symptom if a smaller subset can preserve meaning while reducing redundancy.


## Question 33

### Are pimples and skin darkening giving overlapping signal?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the overlap table is quantifying how PCOS prevalence changes across combined symptom states.
summary_q33 = (
    df.groupby(["pimples_y_n", "skin_darkening_y_n"])["pcos_y_n"]
    .agg(["count", "mean"])
    .rename(columns={"count": "n", "mean": "pcos_prevalence"})
    .reset_index()
)
summary_q33["pcos_prevalence_pct"] = (summary_q33["pcos_prevalence"] * 100).round(1)
summary_q33["pimples_label"] = summary_q33["pimples_y_n"].map({0: "No Pimples", 1: "Pimples"})
summary_q33["skin_darkening_label"] = summary_q33["skin_darkening_y_n"].map({0: "No Skin Darkening", 1: "Skin Darkening"})
display(summary_q33)

heatmap_q33 = summary_q33.pivot(
    index="skin_darkening_label",
    columns="pimples_label",
    values="pcos_prevalence_pct",
)


In [ ]:
# Plotting the overlap heatmap is showing whether the joint symptom state is concentrating PCOS prevalence.
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    heatmap_q33,
    annot=True,
    fmt=".1f",
    cmap="rocket_r",
    linewidths=0.5,
    cbar_kws={"label": "PCOS Prevalence (%)"},
    ax=ax,
)
ax.set_title("Question 33: PCOS Prevalence Across Pimples and Skin Darkening States")
ax.set_xlabel("")
ax.set_ylabel("")
save_figure(fig, "q33_pimples_skin_darkening_overlap_heatmap.png")
plt.show()


### Insight


    This overlap heatmap is showing whether pimples and skin darkening are mostly repeating the same risk signal or whether their combination is marking a particularly high-burden subgroup. If the joint-positive cell is clearly warmer than the single-symptom cells, the pair may be carrying useful combined information even if each feature is individually correlated.


## Derived Feature Analysis

This section is testing simple engineered views of the data that may become useful during later feature-selection and modeling work.


## Question 34

### Does a BMI-category distribution differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the BMI-category table is summarizing how participants are distributing across clinical weight classes.
summary_q34 = (
    pd.crosstab(df["bmi_category"], df["pcos_label"], normalize="columns")
    .mul(100)
    .round(1)
)
display(summary_q34)


In [ ]:
# Plotting the BMI-category distribution is showing whether higher BMI classes are overrepresented among PCOS-positive participants.
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(
    data=df,
    x="bmi_category",
    hue="pcos_label",
    order=["Underweight", "Normal", "Overweight", "Obese"],
    hue_order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 34: BMI Category Distribution by PCOS Status")
ax.set_xlabel("BMI Category")
ax.set_ylabel("Participant Count")
ax.legend(title="")
save_figure(fig, "q34_bmi_category_distribution.png")
plt.show()


### Insight


    The BMI-category view is translating a continuous measure into a screening-friendly clinical lens. If the PCOS-positive group is accumulating more heavily in the overweight and obese categories, that supports keeping BMI both as a raw continuous feature and as a potentially useful engineered categorical marker later on.


## Question 35

### Does a simple non-invasive burden score differ by PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the burden-score summary is quantifying how multiple accessible features are clustering within each PCOS group.
summary_q35 = grouped_numeric_summary(df, "non_invasive_burden_score")
burden_distribution_q35 = (
    df.groupby(["pcos_label", "non_invasive_burden_score"])
    .size()
    .rename("count")
    .reset_index()
)
display(summary_q35)
display(burden_distribution_q35)


In [ ]:
# Plotting the burden score is showing whether symptom accumulation is concentrating in the PCOS-positive cohort.
plot_numeric_by_target(
    data=df,
    feature="non_invasive_burden_score",
    ylabel="Non-Invasive Burden Score",
    title="Question 35: Non-Invasive Burden Score by PCOS Status",
    slug="q35_non_invasive_burden_score.png",
    kind="violin",
)


### Insight


    The burden score is showing whether multiple accessible symptom and lifestyle signals are stacking together in the PCOS-positive group. This matters because a composite burden feature can sometimes capture the syndrome pattern more efficiently than any single symptom considered in isolation.


## Question 36

### What does the 364 versus 177 class split imply for false-negative cost in a screening workflow?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the false-negative illustration table is translating class balance into a clinical screening consequence.
class_counts_q36 = (
    df["pcos_label"]
    .value_counts()
    .reindex(PCOS_LABEL_ORDER)
    .rename_axis("pcos_label")
    .reset_index(name="count")
)
total_positive_q36 = int(class_counts_q36.loc[class_counts_q36["pcos_label"] == "PCOS Positive", "count"].iloc[0])
summary_q36 = pd.DataFrame(
    {
        "assumed_sensitivity": [0.90, 0.80, 0.70, 0.60],
        "false_negative_rate": [0.10, 0.20, 0.30, 0.40],
    }
)
summary_q36["missed_positive_cases"] = (summary_q36["false_negative_rate"] * total_positive_q36).round().astype(int)
display(class_counts_q36)
display(summary_q36)


In [ ]:
# Plotting the missed-case illustration is showing how quickly clinical cost grows when sensitivity drops.
fig, ax = plt.subplots(figsize=(8, 5))
plot_q36 = summary_q36.copy()
plot_q36["false_negative_rate_label"] = (plot_q36["false_negative_rate"] * 100).astype(int).astype(str) + "% FNR"
sns.barplot(
    data=plot_q36,
    x="false_negative_rate_label",
    y="missed_positive_cases",
    palette=["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"],
    ax=ax,
)
for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(
        f"{int(height)}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        xytext=(0, 6),
        textcoords="offset points",
    )
ax.set_title("Question 36: Illustrative False-Negative Cost in the Clinical Cohort")
ax.set_xlabel("Assumed False-Negative Rate")
ax.set_ylabel("Missed PCOS Cases")
save_figure(fig, "q36_false_negative_cost_illustration.png")
plt.show()


### Insight


    The class split is reminding us that sensitivity is clinically important even in a moderately imbalanced dataset. With 177 PCOS-positive cases in this cohort, a false-negative rate of 20% would already correspond to about 35 missed cases, which is why later model evaluation should prioritize recall and clinical miss-cost rather than accuracy alone.


## Observational Summary

The clinical EDA is showing that the most persuasive PCOS signals in this cleaned cohort are clustering around BMI, weight, cycle irregularity codes, symptom burden, and follicle counts. The most visible symptom-level differences are appearing in weight gain, hair growth, skin darkening, and pimples, while general vital signs such as pulse rate and respiratory rate are moving much less.

The cardiometabolic proxies are showing some upward stress in the PCOS-positive cohort, especially through BMI and the upper tail of random blood sugar, but the blood-pressure variables are not separating the groups strongly on their own. This is suggesting that cardiometabolic variables may contribute context rather than headline discrimination inside the PCOS cohort.

The ovarian variables are showing some of the strongest differences, especially follicle counts. That is clinically coherent, but it also reinforces the importance of keeping the later non-invasive modeling phase separate from the broader clinical interpretation phase.


## Feature Selection Insight

### Features to Keep Prominently
- `bmi`
- `weight_kg`
- `weight_gain_y_n`
- `hair_growth_y_n`
- `skin_darkening_y_n`
- `pimples_y_n`
- `cycle_regularity_code`
- `cycle_length_days` as a recorded measure
- `rbs_mg_dl`

### Features to Treat Carefully
- `waist_hip_ratio`, because it is appearing weaker than BMI but may still add nuance
- `fast_food_y_n`, because the signal may be confounded by lifestyle and reporting behavior
- `regular_exercise_y_n`, because the group separation is limited
- `hb_g_dl`, because the biological shift is modest

### Features That Look Weaker or Noisier as Standalone Signals
- `pulse_rate_bpm`
- `respiratory_rate_breaths_min`
- `systolic_bp_mmhg`
- `diastolic_bp_mmhg`

### Features Worth Engineering Later
- `non_invasive_burden_score`
- `bmi_category`
- symptom interaction features such as pimples plus skin darkening
- anthropometric interaction views such as BMI plus waist-hip ratio

### Important Boundary
Ovarian morphology variables such as follicle counts are remaining highly informative for interpretation, but they should be kept separate from strictly non-invasive model sets when the later ablation study is being designed.
